# Celabot — Hello World del pipeline

Tu primer prototipo. Vamos a:

1. Detectar personas en un video usando **YOLO** (carril rápido, barato).
2. Decidir cuándo el video tiene algo que vale la pena revisar (regla simple).
3. Pedirle a **Gemini** que analice ese clip y diga si hay anomalía (carril lento, caro pero inteligente).

Este es el **núcleo conceptual** de Celabot. Todo lo demás (RTSP, nube, WhatsApp, dashboard) es plomería alrededor de este loop.

> Abre este notebook en Google Colab: `File → Upload notebook` y sube `hello.ipynb`, o usa el botón "Open in Colab" si lo tienes en GitHub.

## 0. Setup

**¿Qué instalamos?**

- `ultralytics` → librería oficial de YOLO. Te baja los pesos del modelo automáticamente la primera vez. Maneja PyTorch por debajo.
- `google-genai` → SDK oficial de Google para llamar a Gemini.
- `opencv-python-headless` → para leer/escribir video. "Headless" porque Colab no tiene pantalla; ahorra dependencias gráficas.

La primera vez tarda ~1 min.

In [ ]:
!pip install -q ultralytics google-genai opencv-python-headless

## 1. Tu primer YOLO

**YOLO** (You Only Look Once) es una familia de detectores de objetos. Le das una imagen y te devuelve una lista de:

- **Bounding box**: las coordenadas `(x1, y1, x2, y2)` de un rectángulo alrededor del objeto.
- **Clase**: qué cosa es (`person`, `handbag`, `bottle`, ...). YOLO preentrenado conoce 80 clases del dataset COCO.
- **Confianza**: número entre 0 y 1.

Vamos a cargar el modelo más liviano (`yolo11n`, ~6 MB) y probarlo en una imagen de muestra que viene con la librería. Más adelante usaremos modelos más grandes para más precisión.

> La primera vez, `YOLO('yolo11n.pt')` baja los pesos. Si ves un progreso de descarga, es normal.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

# Imagen de muestra que viene con ultralytics
results = model('https://ultralytics.com/images/bus.jpg', verbose=False)

# results es una lista (una entrada por imagen). Inspeccionamos la primera.
r = results[0]
print(f"Detecciones encontradas: {len(r.boxes)}")
for box in r.boxes:
    cls_id = int(box.cls[0])
    cls_name = r.names[cls_id]
    conf = float(box.conf[0])
    xyxy = box.xyxy[0].tolist()
    print(f"  {cls_name:10s} conf={conf:.2f} box={[round(v) for v in xyxy]}")

**Qué acabas de ver:** una imagen con un bus + personas, y YOLO te listó cada objeto con su clase y confianza. **20–40 ms por imagen** en Colab. Esto es el "carril rápido" del que hablamos en el plan.

Si quieres ver la imagen anotada:

In [ ]:
from IPython.display import Image, display
import cv2

annotated = r.plot()  # numpy array BGR con cajas dibujadas
cv2.imwrite('annotated.jpg', annotated)
display(Image('annotated.jpg'))

## 2. Sube tu video

Sube un video corto (10–60 s, MP4 idealmente) de cualquier escena: una tienda, gente caminando, tú simulando que escondes algo. La calidad de teléfono está bien.

Ejecuta la celda y aparecerá un botón para escoger el archivo.

In [ ]:
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print(f"Video listo: {VIDEO_PATH}")

## 3. Carril rápido: muestreo de frames

**Idea:** un video de 30 fps tiene 1800 frames en un minuto. Correr YOLO en todos es desperdicio — frames consecutivos son casi idénticos. Muestreamos **1 cada N frames** (típicamente cada 0.2–0.5 segundos).

Por cada frame muestreado, registramos cuántas personas detecta YOLO. Eso nos da una **serie temporal de actividad** que nos dirá dónde mirar más de cerca.

In [ ]:
import cv2

SAMPLE_EVERY_SECONDS = 0.3  # muestreamos un frame cada 0.3 s
PERSON_CLASS_ID = 0          # 'person' en COCO es la clase 0

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
step = max(1, int(fps * SAMPLE_EVERY_SECONDS))
duration = total_frames / fps

print(f"Video: {duration:.1f} s, {fps:.1f} fps, {total_frames} frames")
print(f"Muestreando 1 cada {step} frames (~{SAMPLE_EVERY_SECONDS}s)\n")

samples = []  # lista de (timestamp_segundos, num_personas)

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % step == 0:
        results = model(frame, verbose=False, classes=[PERSON_CLASS_ID])
        n_persons = len(results[0].boxes)
        ts = frame_idx / fps
        samples.append((ts, n_persons))
    frame_idx += 1

cap.release()

print(f"Muestras tomadas: {len(samples)}")
print("Primeras 10 muestras (segundo, #personas):")
for s in samples[:10]:
    print(f"  t={s[0]:5.2f}s  personas={s[1]}")

**Qué tienes ahora:** una lista de "a tal segundo había N personas". Por sí sola no detecta nada. Es la **materia prima** que el siguiente paso usa para decidir si vale la pena gastar dinero en el VLM.

## 4. Regla: ¿qué cuenta como "candidato"?

Acá vive la inteligencia barata del sistema. En producción esta capa tendrá decenas de reglas (pose, mano-objeto, tracking, audio...). Para el Hello World usamos **una regla mínima**:

> "Hay un candidato si hubo al menos una persona durante ≥ 2 segundos seguidos."

Esta regla por sí sola es tonta — no detecta robos — pero sí decide **qué ventana de video le mostramos al VLM**, que es el que sí razona. Más adelante reemplazamos esta regla por algo serio (concealment, arma, etc).

El resultado es una lista de **ventanas candidatas**: `(t_inicio, t_fin)` en segundos.

In [ ]:
MIN_PERSONS = 1
MIN_DURATION_SECONDS = 2.0

windows = []
window_start = None

for ts, n in samples:
    if n >= MIN_PERSONS:
        if window_start is None:
            window_start = ts
        last_ts = ts
    else:
        if window_start is not None and last_ts - window_start >= MIN_DURATION_SECONDS:
            windows.append((window_start, last_ts))
        window_start = None

# Cerrar la última ventana si quedó abierta
if window_start is not None and last_ts - window_start >= MIN_DURATION_SECONDS:
    windows.append((window_start, last_ts))

print(f"Ventanas candidatas: {len(windows)}")
for w in windows:
    print(f"  {w[0]:.1f}s → {w[1]:.1f}s  (duración {w[1]-w[0]:.1f}s)")

## 5. Extraer el clip de la primera ventana

Para no abusar de la API del VLM, en este Hello World analizamos solo la **primera ventana** y la limitamos a ~10 s. En producción mandarías cada ventana en paralelo con *rate limiting*.

Usamos `ffmpeg` (ya viene en Colab). Re-codificamos a H.264 con corte exacto: `-ss` después de `-i` da precisión de cuadro y `-c:v libx264 -preset ultrafast` re-comprime rápido. Si usaras `-c copy` tendrías un corte mal alineado cuando el `start` no cae en un *keyframe*.

In [ ]:
import subprocess, os

if not windows:
    print("No hay ventanas candidatas. Sube otro video o baja MIN_DURATION_SECONDS.")
else:
    start, end = windows[0]
    end = min(end, start + 10)  # cap a 10s
    clip_path = 'candidate.mp4'
    if os.path.exists(clip_path):
        os.remove(clip_path)
    cmd = [
        'ffmpeg', '-y', '-loglevel', 'error',
        '-i', VIDEO_PATH,
        '-ss', f'{start:.2f}',
        '-t', f'{end - start:.2f}',
        '-c:v', 'libx264', '-preset', 'ultrafast',
        '-movflags', '+faststart',
        '-an',
        clip_path,
    ]
    subprocess.run(cmd, check=True)
    print(f"Clip extraído: {clip_path} ({os.path.getsize(clip_path)/1024:.0f} KB)")

## 6. Carril lento: Gemini analiza el clip

Ahora viene la parte cara pero inteligente. Le mandamos el clip a **Gemini 2.5 Flash** (rápido y económico de los modelos de Google) con un *prompt* estructurado.

**Conceptos importantes del prompt:**

1. **Rol** ("Eres un analista de seguridad..."): orienta al modelo.
2. **Pregunta observable** (no de intención): preguntamos por *hechos visibles* — ocultar algo, agresión, arma — no por "¿iba a robar?". Esto es lo que discutimos en §17.1 del plan.
3. **Esquema JSON estricto**: para que la salida sea procesable por código, no prosa.
4. **Anti-falsos-positivos explícitos**: le decimos qué *no* contar como anomalía (revisar etiqueta, sacar el celular).

**API key de Gemini:** consíguela gratis en https://aistudio.google.com/apikey. Cópiala y pégala cuando te la pida.

In [ ]:
import getpass
GEMINI_API_KEY = getpass.getpass('Pega tu API key de Gemini (no se mostrará): ')

In [ ]:
from google import genai
import time, json

client = genai.Client(api_key=GEMINI_API_KEY)

PROMPT = '''Eres un analista de seguridad de una tienda pequeña en Colombia.
Mira este clip de cámara de seguridad y responde ESTRICTAMENTE en JSON con este esquema:

{
  "anomaly": boolean,
  "confidence": number,           // 0.0 a 1.0
  "category": string,             // "ocultamiento" | "agresion" | "arma" | "intrusion" | "merodeo" | "normal"
  "observed_behaviors": string[], // hechos visibles, no interpretaciones
  "reason": string,               // 1-2 frases en español
  "suggested_action": string      // qué debería hacer el dueño
}

Reglas:
- Pregúntate por hechos OBSERVABLES, no por intención. No digas "parecía sospechoso"; di "metió un objeto en el bolso".
- Estos comportamientos son NORMALES (no marques anomalía):
  revisar etiqueta de un producto, sacar el celular, buscar la billetera,
  hablar con otra persona, mirar el menú o las góndolas.
- Solo marca anomalía si ves: ocultar producto en ropa/bolso sin pagar, agresión física,
  arma visible, persona en horario cerrado, o conducta de grab-and-run.
- Responde SOLO con el JSON, sin texto adicional ni bloques de código.
'''

# Subimos el clip
print('Subiendo clip a Gemini...')
video_file = client.files.upload(file=clip_path)

# Esperamos a que esté procesado
while video_file.state.name == 'PROCESSING':
    time.sleep(2)
    video_file = client.files.get(name=video_file.name)
print(f'Clip listo (estado: {video_file.state.name})')

# Llamamos al modelo
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=[video_file, PROMPT],
)

raw = response.text.strip()
# Limpiar si vino con bloque markdown a pesar del prompt
if raw.startswith('```'):
    raw = raw.split('```')[1]
    if raw.startswith('json'):
        raw = raw[4:]
    raw = raw.strip()

try:
    verdict = json.loads(raw)
    print('\n=== VEREDICTO ===')
    print(json.dumps(verdict, indent=2, ensure_ascii=False))
except json.JSONDecodeError:
    print('No se pudo parsear como JSON. Respuesta cruda:')
    print(raw)

## 7. ¿Qué acaba de pasar?

Reconstruyamos el viaje:

1. YOLO procesó tu video en milisegundos por frame → 100% local, 0$ de API.
2. Una regla simple (al menos 1 persona ≥ 2 s) extrajo ventanas candidatas.
3. Solo de esas ventanas pedimos opinión a Gemini → 1 llamada a la API en vez de N miles.
4. Gemini devolvió un **JSON estructurado** que tu backend podría usar para decidir si dispara una alerta WhatsApp o no.

Esto es exactamente el patrón **carril rápido + carril lento** del plan §3.3. Lo que cambia en producción:

- En vez de un MP4, es un stream RTSP en vivo.
- La "regla simple" se vuelve un conjunto rico (pose, manos, arma, audio).
- El JSON de Gemini va a Redis, dispara WhatsApp, persiste en Postgres, etc.
- Las etiquetas del dueño ("era falsa alarma") regresan como *feedback* para entrenar modelos propios (§17 del plan).

## Experimentos sugeridos

- Bajar `SAMPLE_EVERY_SECONDS` a `0.1` y subir `MIN_PERSONS` a `2`.
- Cambiar `yolo11n.pt` por `yolo11s.pt` o `yolo11m.pt` (más grandes, más precisos, más lentos).
- Editar el `PROMPT` para enfocarse en un solo caso y ver cómo cambia el veredicto.
- Subir dos videos distintos y comparar — uno "normal", uno con comportamiento sospechoso.

## Siguiente paso

Abre `prototype/zones.ipynb`. Ahí reemplazamos esta "regla mínima" por **tracking + zonas + cruce de línea** con la librería [Supervision](https://github.com/roboflow/supervision). Construyes detecciones reales de **intrusión, merodeo y cruce de salida** sobre el mismo video.